In [45]:
import pandas as pd
import numpy as np

import re
import unicodedata
from collections import defaultdict, Counter
from pathlib import Path

from anyascii import anyascii
from rapidfuzz.distance import Levenshtein

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path("sampled_data")

S1_FILE = BASE_DIR / "sample_source1.tsv"
S2_FILE = BASE_DIR / "sample_source2.tsv"
S3_FILE = BASE_DIR / "sample_source3.tsv"
GT_FILE = BASE_DIR / "sample_ground_truth.tsv"

OUTPUT_DIR = Path("blocking_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADDRESS_OVERLAP_THRESHOLD = 2

# Very common address words should not drive blocking.
WEAK_ADDRESS_TOKENS = {
    "road", "rd",
    "street", "st",
    "avenue", "ave",
    "lane", "ln",
    "drive", "dr",
    "highway", "hwy",
    "boulevard", "blvd",
    "way",
    "place", "pl",
    "parkway", "pkwy",
    "unit",
    "suite", "ste",
    "apt", "apartment",
    "floor",
    "fl",
    "building", "bldg",
    "block",
    "district",
    "city",
    "state",
    "county",
    "india",
    "usa",
    "us",
}

MIN_TOKEN_LENGTH = 3

print("Configuration loaded.")
print(f"Address overlap threshold: {ADDRESS_OVERLAP_THRESHOLD}")
print(f"Weak address tokens: {len(WEAK_ADDRESS_TOKENS)}")

Configuration loaded.
Address overlap threshold: 2
Weak address tokens: 36


In [46]:
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

s1 = pd.read_csv(S1_FILE, sep="\t", dtype=str).fillna("")
s2 = pd.read_csv(S2_FILE, sep="\t", dtype=str).fillna("")
s3 = pd.read_csv(S3_FILE, sep="\t", dtype=str).fillna("")
gt = pd.read_csv(GT_FILE, sep="\t", dtype=str).fillna("")

print(f"S1 shape: {s1.shape}")
print(f"S2 shape: {s2.shape}")
print(f"S3 shape: {s3.shape}")
print(f"GT shape: {gt.shape}")

print("\nColumns:")
print("S1:", list(s1.columns))
print("S2:", list(s2.columns))
print("S3:", list(s3.columns))
print("GT:", list(gt.columns))


LOADING DATA
S1 shape: (1000, 4)
S2 shape: (9864, 4)
S3 shape: (10842, 4)
GT shape: (1000, 2)

Columns:
S1: ['entity_id', 'business_name', 'business_address', 'country']
S2: ['entity_id', 'business_name', 'business_address', 'country']
S3: ['entity_id', 'business_name', 'business_address', 'country']
GT: ['source1_entity_id', 'matched_entity_ids']


In [47]:
def is_latin_char(ch):
    """
    Returns True if the character belongs to the Latin script.
    """
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return False

    return "LATIN" in name


def contains_non_latin(text):
    """
    Detect whether text contains alphabetic characters
    outside the Latin script.
    """
    for ch in text:
        if ch.isalpha() and not is_latin_char(ch):
            return True
    return False


def transliterate_text(text):
    """
    Convert non-Latin text into a Latin-oriented representation.

    Original text is preserved elsewhere.
    """
    if not text:
        return ""

    return anyascii(text)


def normalize_text(text, transliterate=False):
    """
    General normalization.

    Steps:
    - Unicode normalization
    - lowercase
    - optional transliteration
    - punctuation -> spaces
    - collapse whitespace
    """
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()

    if transliterate:
        text = transliterate_text(text)

    # Keep letters and numbers, replace everything else with spaces
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text):
    """
    Basic normalized whitespace tokenization.
    """
    if not text:
        return []

    return text.split()


def normalize_token(token):
    return normalize_text(token, transliterate=False)


def transliterated_tokens(text):
    """
    Tokenize first, then transliterate tokens that contain
    non-Latin characters.
    """
    result = []

    for token in tokenize(normalize_text(text)):
        if contains_non_latin(token):
            token = transliterate_text(token)

        token = normalize_text(token, transliterate=False)

        if token:
            result.append(token)

    return result

In [48]:
def prepare_source(df):
    df = df.copy()

    df["name_norm"] = df["business_name"].apply(
        lambda x: normalize_text(x, transliterate=False)
    )

    df["address_norm"] = df["business_address"].apply(
        lambda x: normalize_text(x, transliterate=False)
    )

    df["name_translit"] = df["business_name"].apply(
        lambda x: normalize_text(x, transliterate=True)
    )

    df["address_translit"] = df["business_address"].apply(
        lambda x: normalize_text(x, transliterate=True)
    )

    df["country_norm"] = df["country"].apply(
        lambda x: normalize_text(x, transliterate=True)
    )

    return df


print("Preparing normalized representations...")

s1 = prepare_source(s1)
s2 = prepare_source(s2)
s3 = prepare_source(s3)

print("Normalization complete.")

Preparing normalized representations...
Normalization complete.


In [49]:
non_latin_s2 = s2[
    s2["business_name"].apply(contains_non_latin) |
    s2["business_address"].apply(contains_non_latin)
]

non_latin_s3 = s3[
    s3["business_name"].apply(contains_non_latin) |
    s3["business_address"].apply(contains_non_latin)
]

print(f"S2 records containing non-Latin text: {len(non_latin_s2)}")
print(f"S3 records containing non-Latin text: {len(non_latin_s3)}")

print("\nS2 transliteration examples:")
print(
    non_latin_s2[
        ["business_name", "name_translit",
         "business_address", "address_translit"]
    ].head(10).to_string(index=False)
)

S2 records containing non-Latin text: 1598
S3 records containing non-Latin text: 1389

S2 transliteration examples:
                               business_name                           name_translit                                                                                                     business_address                                                                                              address_translit
                பிளாக் ஐடி பிரைவேட் லிமிடெட்             pilak aiti piraivet limitet                                                                                 SF.NO364/IB2, COIMBATORE, Tamil Nadu                                                                            sf no364 ib2 coimbatore tamil nadu
      नॉर्थ इंफ्रा ट्रेडिंग प्राइवेट लिमिटेड    north imphra tredimg praivet limited                                                                             NO. 634 UNIT NO.- 938, WEST DELHI, Delhi                                                           

In [50]:
def get_informative_address_tokens(address):
    """
    Extract address tokens that are useful for blocking.

    Uses transliterated representation so non-Latin addresses
    can participate in the same Latin-oriented index.
    """

    tokens = transliterated_tokens(address)

    informative = set()

    for token in tokens:

        if len(token) < MIN_TOKEN_LENGTH:
            continue

        if token in WEAK_ADDRESS_TOKENS:
            continue

        # Ignore tokens that are purely punctuation/noise
        if not re.search(r"[a-z0-9]", token):
            continue

        informative.add(token)

    return informative

In [51]:
s1["address_tokens"] = s1["business_address"].apply(
    get_informative_address_tokens
)

s2["address_tokens"] = s2["business_address"].apply(
    get_informative_address_tokens
)

s3["address_tokens"] = s3["business_address"].apply(
    get_informative_address_tokens
)

print("Example S1 address tokens:")

for _, row in s1.head(10).iterrows():
    print(row["entity_id"])
    print(row["address_tokens"])
    print()

Example S1 address tokens:
S1-53356671
{'4850', 'otisco'}

S1-320151505
{'19034', 'woodburn'}

S1-938947364
{'7241', 'mesa', 'osage'}

S1-195839862
{'thiruvananthapuram', 'prabhul', 'kerala', 'karimbalur', 'cottage', 'ayiroor', '606', 'elakamon', 'trivandrum'}

S1-655046555
{'charleswood', 'memphis', '4255'}

S1-758070915
{'virginia', '1641', 'hueytown'}

S1-169338169
{'sanford', 'guillemette'}

S1-796421498
{'jankalyan', '1204', 'oasis', '12thfloor', 'malad', 'mumbai', 'billabong', 'nagarmalvani', 'roya', 'school', 'maharashtra'}

S1-802535179
{'wood', '2105', 'cove', 'ridge', 'park', 'cedar'}

S1-97176033
{'301', 'sai', 'thane', 'dham', 'new', 'park', 'ramdev', 'chsl', 'maharashtra'}



In [52]:
def build_address_index(df):
    index = defaultdict(set)

    for _, row in df.iterrows():

        entity_id = row["entity_id"]

        for token in row["address_tokens"]:
            index[token].add(entity_id)

    return index


print("=" * 70)
print("BUILDING ADDRESS INDEX")
print("=" * 70)

s2_address_index = build_address_index(s2)
s3_address_index = build_address_index(s3)

print(f"S2 address index tokens: {len(s2_address_index)}")
print(f"S3 address index tokens: {len(s3_address_index)}")

BUILDING ADDRESS INDEX
S2 address index tokens: 16008
S3 address index tokens: 16118


In [53]:
def build_entity_lookup(df):
    return df.set_index("entity_id").to_dict("index")


s2_lookup = build_entity_lookup(s2)
s3_lookup = build_entity_lookup(s3)


def generate_address_candidates_for_s1(
    s1_row,
    address_index,
    entity_lookup,
    source_name
):
    """
    Generate address-block candidates for one S1 record.
    """

    s1_country = s1_row["country_norm"]
    s1_tokens = s1_row["address_tokens"]

    token_counts = Counter()

    for token in s1_tokens:

        candidate_ids = address_index.get(token, set())

        for candidate_id in candidate_ids:
            token_counts[candidate_id] += 1

    candidates = []

    for candidate_id, overlap_count in token_counts.items():

        candidate = entity_lookup[candidate_id]

        if candidate["country_norm"] != s1_country:
            continue

        if overlap_count < ADDRESS_OVERLAP_THRESHOLD:
            continue

        candidates.append({
            "s1_entity_id": s1_row["entity_id"],
            "candidate_entity_id": candidate_id,
            "candidate_source": source_name,
            "block_type": "address",
            "shared_address_tokens": overlap_count
        })

    return candidates

In [54]:
print("=" * 70)
print("RUNNING ADDRESS BLOCKER")
print("=" * 70)

candidate_rows = []

for i, (_, row) in enumerate(s1.iterrows(), start=1):

    if i % 100 == 0 or i == 1:
        print(f"Processing S1 {i}/{len(s1)}")

    # S2
    candidate_rows.extend(
        generate_address_candidates_for_s1(
            row,
            s2_address_index,
            s2_lookup,
            "S2"
        )
    )

    # S3
    candidate_rows.extend(
        generate_address_candidates_for_s1(
            row,
            s3_address_index,
            s3_lookup,
            "S3"
        )
    )

address_candidates = pd.DataFrame(candidate_rows)

print("\nAddress blocking complete.")
print(f"Candidate rows: {len(address_candidates):,}")

if len(address_candidates):
    print(
        "Unique S1 entities with candidates:",
        address_candidates["s1_entity_id"].nunique()
    )

RUNNING ADDRESS BLOCKER
Processing S1 1/1000
Processing S1 100/1000
Processing S1 200/1000
Processing S1 300/1000
Processing S1 400/1000
Processing S1 500/1000
Processing S1 600/1000
Processing S1 700/1000
Processing S1 800/1000
Processing S1 900/1000
Processing S1 1000/1000

Address blocking complete.
Candidate rows: 100,083
Unique S1 entities with candidates: 959


In [55]:
address_candidates = (
    address_candidates
    .drop_duplicates(
        subset=[
            "s1_entity_id",
            "candidate_entity_id"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Unique S1-candidate pairs: "
    f"{len(address_candidates):,}"
)

Unique S1-candidate pairs: 100,083


In [56]:
def parse_ground_truth(gt_df):
    rows = []

    for _, row in gt_df.iterrows():

        s1_id = row["source1_entity_id"]
        matched = str(row["matched_entity_ids"]).strip()

        if not matched:
            continue

        for candidate_id in matched.split(","):

            candidate_id = candidate_id.strip()

            if candidate_id:
                rows.append({
                    "s1_entity_id": s1_id,
                    "true_entity_id": candidate_id
                })

    return pd.DataFrame(rows)


gt_pairs = parse_ground_truth(gt)

print(f"Ground-truth positive pairs: {len(gt_pairs):,}")

print(
    "Ground-truth S1 entities with at least one match:",
    gt_pairs["s1_entity_id"].nunique()
)

Ground-truth positive pairs: 3,451
Ground-truth S1 entities with at least one match: 934


In [57]:
candidate_pairs = address_candidates[
    [
        "s1_entity_id",
        "candidate_entity_id"
    ]
].drop_duplicates()

evaluation = gt_pairs.merge(
    candidate_pairs,
    left_on=[
        "s1_entity_id",
        "true_entity_id"
    ],
    right_on=[
        "s1_entity_id",
        "candidate_entity_id"
    ],
    how="left",
    indicator=True
)

evaluation["retrieved"] = (
    evaluation["_merge"] == "both"
)

retrieved = evaluation["retrieved"].sum()
total_true = len(evaluation)

blocking_recall = (
    retrieved / total_true
    if total_true > 0
    else 0
)

print("=" * 70)
print("BLOCKING RECALL")
print("=" * 70)

print(f"True matching pairs: {total_true:,}")
print(f"Retrieved by blocking: {retrieved:,}")
print(f"Missed true pairs: {total_true - retrieved:,}")
print(f"Blocking recall: {blocking_recall:.4%}")

BLOCKING RECALL
True matching pairs: 3,451
Retrieved by blocking: 3,180
Missed true pairs: 271
Blocking recall: 92.1472%


In [58]:
candidate_counts = (
    candidate_pairs
    .groupby("s1_entity_id")
    .size()
)

print("=" * 70)
print("CANDIDATE VOLUME")
print("=" * 70)

print(f"Total candidate pairs: {len(candidate_pairs):,}")

print(
    f"Average candidates/S1: "
    f"{candidate_counts.mean():.2f}"
)

print(
    f"Median candidates/S1: "
    f"{candidate_counts.median():.2f}"
)

print(
    f"P95 candidates/S1: "
    f"{candidate_counts.quantile(0.95):.2f}"
)

print(
    f"P99 candidates/S1: "
    f"{candidate_counts.quantile(0.99):.2f}"
)

print(
    f"Maximum candidates/S1: "
    f"{candidate_counts.max()}"
)

total_possible = len(s1) * (len(s2) + len(s3))

reduction_ratio = (
    1 - len(candidate_pairs) / total_possible
    if total_possible > 0
    else 0
)

print(f"\nTotal possible Cartesian pairs: {total_possible:,}")
print(f"Candidate reduction ratio: {reduction_ratio:.4%}")

CANDIDATE VOLUME
Total candidate pairs: 100,083
Average candidates/S1: 104.36
Median candidates/S1: 7.00
P95 candidates/S1: 452.30
P99 candidates/S1: 775.14
Maximum candidates/S1: 914

Total possible Cartesian pairs: 20,706,000
Candidate reduction ratio: 99.5166%


In [59]:
missed_pairs = evaluation[
    ~evaluation["retrieved"]
][
    [
        "s1_entity_id",
        "true_entity_id"
    ]
].copy()

print("=" * 70)
print("MISSED TRUE PAIRS")
print("=" * 70)

print(f"Missed pairs: {len(missed_pairs):,}")

missed_pairs.to_csv(
    OUTPUT_DIR / "missed_blocking_pairs.tsv",
    sep="\t",
    index=False
)

print(
    f"Saved to: "
    f"{OUTPUT_DIR / 'missed_blocking_pairs.tsv'}"
)

MISSED TRUE PAIRS
Missed pairs: 271
Saved to: blocking_results\missed_blocking_pairs.tsv


In [60]:
all_source_records = pd.concat(
    [
        s2.assign(source="S2"),
        s3.assign(source="S3")
    ],
    ignore_index=True
)

s1_lookup = s1.set_index("entity_id")
candidate_lookup = all_source_records.set_index("entity_id")


for _, pair in missed_pairs.head(20).iterrows():

    s1_id = pair["s1_entity_id"]
    true_id = pair["true_entity_id"]

    s1_row = s1_lookup.loc[s1_id]
    true_row = candidate_lookup.loc[true_id]

    print("=" * 70)
    print("MISSED PAIR")
    print("=" * 70)

    print("S1:", s1_id)
    print("Name:", s1_row["business_name"])
    print("Address:", s1_row["business_address"])
    print("Country:", s1_row["country"])

    print()

    print("TRUE:", true_id)
    print("Name:", true_row["business_name"])
    print("Address:", true_row["business_address"])
    print("Country:", true_row["country"])

    print()

MISSED PAIR
S1: S1-511076246
Name: Agra Granites
Address: Flat No-4, Sector-4B, Vaishno Plaza Awas Vikas Colony Sikandra, Agra, Uttar Pradesh
Country: India

TRUE: S2-100727491
Name: Agra Granites [Limited]
Address: 
Country: India

MISSED PAIR
S1: S1-210935903
Name: Leann Colon Federal Acquisitions Inc
Address: 9229 106th Way, Fl 1, Scottsdale, AZ
Country: US

TRUE: S3-121609463
Name: Leann Colon  Federal
Address: 
Country: US

MISSED PAIR
S1: S1-10580196
Name: Secure Telecom
Address: 310 Calion Street, Unit Apt C, Jonesboro, AR
Country: US

TRUE: S2-34357118
Name: SECURE TELECOM INC.
Address: 
Country: US

MISSED PAIR
S1: S1-748867572
Name: Britt Best Willow LLC
Address: 16955 Toronto Avenue, Unit Apartment 115, Prior Lake, MN
Country: US

TRUE: S3-726962279
Name: britt best willow willow llc
Address: 
Country: US

MISSED PAIR
S1: S1-651483834
Name: IX Innovative Abony Corp
Address: 40 Savage Lane, NC, Hendersonville
Country: US

TRUE: S2-453351217
Name: Corp IX Innovative Abony
Addr

## Multi-strategy candidate generation

Address-token blocking only retrieves records with exact shared address tokens. The following methods improve recall for spelling errors, abbreviations, word-order changes, transliteration, and formatting differences:

- Character n-grams
- Token Jaccard similarity
- Levenshtein similarity
- TF-IDF cosine similarity

Candidates from all methods are combined and evaluated against the ground truth.

In [61]:
from rapidfuzz import process, fuzz

NGRAM_SIZE = 3
NGRAM_TOP_K = 30
FUZZY_TOP_K = 30
COSINE_TOP_K = 30

JACCARD_THRESHOLD = 0.25
LEVENSHTEIN_THRESHOLD = 0.60
COSINE_THRESHOLD = 0.35


def blocking_text(row):
    return normalize_text(
        f"{row['business_name']} {row['business_address']}",
        transliterate=True
    )


def char_ngrams(text, n=NGRAM_SIZE):
    text = f" {text} "
    return {
        text[i:i + n]
        for i in range(max(1, len(text) - n + 1))
    }


def jaccard_similarity(left, right):
    left = set(left)
    right = set(right)

    if not left and not right:
        return 1.0

    if not left or not right:
        return 0.0

    return len(left & right) / len(left | right)


all_source_records = pd.concat(
    [
        s2.assign(source="S2"),
        s3.assign(source="S3")
    ],
    ignore_index=True
).copy()

s1 = s1.copy()

s1["blocking_text"] = s1.apply(blocking_text, axis=1)
all_source_records["blocking_text"] = (
    all_source_records.apply(blocking_text, axis=1)
)

s1["blocking_tokens"] = s1["blocking_text"].apply(tokenize)
all_source_records["blocking_tokens"] = (
    all_source_records["blocking_text"].apply(tokenize)
)

candidate_lookup = (
    all_source_records
    .set_index("entity_id")
    .to_dict("index")
)

In [62]:
ngram_index = defaultdict(set)
token_index = defaultdict(set)

for _, row in all_source_records.iterrows():
    entity_id = row["entity_id"]

    for gram in char_ngrams(row["blocking_text"]):
        ngram_index[gram].add(entity_id)

    for token in set(row["blocking_tokens"]):
        token_index[token].add(entity_id)


def generate_ngram_candidates(row):
    counts = Counter()

    for gram in char_ngrams(row["blocking_text"]):
        for entity_id in ngram_index.get(gram, set()):
            counts[entity_id] += 1

    query_ngrams = char_ngrams(row["blocking_text"])
    results = []

    for entity_id, _ in counts.most_common(NGRAM_TOP_K):
        candidate = candidate_lookup[entity_id]

        if candidate["country_norm"] != row["country_norm"]:
            continue

        candidate_ngrams = char_ngrams(candidate["blocking_text"])
        score = len(query_ngrams & candidate_ngrams) / max(
            len(query_ngrams | candidate_ngrams),
            1
        )

        results.append({
            "s1_entity_id": row["entity_id"],
            "candidate_entity_id": entity_id,
            "method": "char_ngram",
            "score": score
        })

    return results


def generate_jaccard_candidates(row):
    possible_ids = set()

    for token in set(row["blocking_tokens"]):
        possible_ids.update(token_index.get(token, set()))

    results = []

    for entity_id in possible_ids:
        candidate = candidate_lookup[entity_id]

        if candidate["country_norm"] != row["country_norm"]:
            continue

        score = jaccard_similarity(
            row["blocking_tokens"],
            candidate["blocking_tokens"]
        )

        if score >= JACCARD_THRESHOLD:
            results.append({
                "s1_entity_id": row["entity_id"],
                "candidate_entity_id": entity_id,
                "method": "jaccard",
                "score": score
            })

    return results

In [63]:
country_choices = defaultdict(dict)

for entity_id, candidate in candidate_lookup.items():
    country_choices[candidate["country_norm"]][entity_id] = (
        candidate["blocking_text"]
    )


def generate_levenshtein_candidates(row):
    choices = country_choices[row["country_norm"]]

    matches = process.extract(
        row["blocking_text"],
        choices,
        scorer=fuzz.ratio,
        limit=FUZZY_TOP_K
    )

    results = []

    for _, score, entity_id in matches:
        normalized_score = score / 100.0

        if normalized_score >= LEVENSHTEIN_THRESHOLD:
            results.append({
                "s1_entity_id": row["entity_id"],
                "candidate_entity_id": entity_id,
                "method": "levenshtein",
                "score": normalized_score
            })

    return results


tfidf_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1
)

source_matrix = tfidf_vectorizer.fit_transform(
    all_source_records["blocking_text"]
)

source_countries = all_source_records["country_norm"].to_numpy()


def generate_cosine_candidates(row):
    query_vector = tfidf_vectorizer.transform(
        [row["blocking_text"]]
    )

    scores = cosine_similarity(
        query_vector,
        source_matrix
    ).ravel()

    valid_indices = np.where(
        source_countries == row["country_norm"]
    )[0]

    ranked_indices = valid_indices[
        np.argsort(scores[valid_indices])[::-1]
    ][:COSINE_TOP_K]

    results = []

    for index in ranked_indices:
        score = float(scores[index])

        if score >= COSINE_THRESHOLD:
            results.append({
                "s1_entity_id": row["entity_id"],
                "candidate_entity_id": (
                    all_source_records.iloc[index]["entity_id"]
                ),
                "method": "cosine",
                "score": score
            })

    return results

In [64]:
method_rows = []

for i, (_, row) in enumerate(s1.iterrows(), start=1):
    if i % 100 == 0 or i == 1:
        print(f"Processing: {i}/{len(s1)}")

    method_rows.extend(generate_ngram_candidates(row))
    method_rows.extend(generate_jaccard_candidates(row))
    method_rows.extend(generate_levenshtein_candidates(row))
    method_rows.extend(generate_cosine_candidates(row))

multi_method_candidates = pd.DataFrame(method_rows)

if multi_method_candidates.empty:
    multi_method_candidates = pd.DataFrame(
        columns=[
            "s1_entity_id",
            "candidate_entity_id",
            "methods",
            "max_score",
            "method_count"
        ]
    )
else:
    multi_method_candidates = (
        multi_method_candidates
        .groupby(
            ["s1_entity_id", "candidate_entity_id"],
            as_index=False
        )
        .agg(
            methods=("method", lambda x: ",".join(sorted(set(x)))),
            max_score=("score", "max"),
            method_count=("method", "nunique")
        )
    )

address_pairs = address_candidates[
    ["s1_entity_id", "candidate_entity_id"]
].drop_duplicates()

multi_method_candidates = pd.concat(
    [
        multi_method_candidates,
        address_pairs.assign(
            methods="address",
            max_score=np.nan,
            method_count=1
        )
    ],
    ignore_index=True
).drop_duplicates(
    subset=["s1_entity_id", "candidate_entity_id"]
)

print(f"Combined candidate pairs: {len(multi_method_candidates):,}")

Processing: 1/1000
Processing: 100/1000
Processing: 200/1000
Processing: 300/1000
Processing: 400/1000
Processing: 500/1000
Processing: 600/1000
Processing: 700/1000
Processing: 800/1000
Processing: 900/1000
Processing: 1000/1000
Combined candidate pairs: 118,014


## Evaluate recall and candidate volume

A good blocker should retrieve as many true pairs as possible while keeping the candidate set small. The main metric is blocking recall:

`retrieved true pairs / total true pairs`

In [65]:
def evaluate_blocker(candidate_df, method_name):
    pairs = candidate_df[
        ["s1_entity_id", "candidate_entity_id"]
    ].drop_duplicates()

    result = gt_pairs.merge(
        pairs,
        left_on=["s1_entity_id", "true_entity_id"],
        right_on=["s1_entity_id", "candidate_entity_id"],
        how="left",
        indicator=True
    )

    retrieved = (result["_merge"] == "both").sum()
    total = len(gt_pairs)

    return {
        "method": method_name,
        "true_pairs": total,
        "retrieved_pairs": int(retrieved),
        "recall": retrieved / total if total else 0,
        "candidate_pairs": len(pairs)
    }


comparison = [
    evaluate_blocker(address_candidates, "address"),
    evaluate_blocker(multi_method_candidates, "combined")
]

blocking_comparison = pd.DataFrame(comparison)
blocking_comparison["recall_percent"] = (
    blocking_comparison["recall"] * 100
).round(2)

blocking_comparison

,method,true_pairs,retrieved_pairs,recall,candidate_pairs,recall_percent
0,address,3451,3180,0.921472,100083,92.15
1,combined,3451,3424,0.992176,118014,99.22


In [66]:
combined_pairs = multi_method_candidates[
    ["s1_entity_id", "candidate_entity_id"]
].drop_duplicates()

combined_evaluation = gt_pairs.merge(
    combined_pairs,
    left_on=["s1_entity_id", "true_entity_id"],
    right_on=["s1_entity_id", "candidate_entity_id"],
    how="left",
    indicator=True
)

missed_multi_method = combined_evaluation[
    combined_evaluation["_merge"] != "both"
][
    ["s1_entity_id", "true_entity_id"]
]

print(
    f"Missed combined pairs: "
    f"{len(missed_multi_method):,}"
)

missed_multi_method.to_csv(
    OUTPUT_DIR / "missed_multi_method_pairs.tsv",
    sep="\t",
    index=False
)

Missed combined pairs: 27


In [67]:
all_source_records = pd.concat(
    [
        s2.assign(source="S2"),
        s3.assign(source="S3")
    ],
    ignore_index=True
)

s1_lookup = s1.set_index("entity_id")
candidate_lookup = all_source_records.set_index("entity_id")


for _, pair in missed_pairs.head(20).iterrows():

    s1_id = pair["s1_entity_id"]
    true_id = pair["true_entity_id"]

    s1_row = s1_lookup.loc[s1_id]
    true_row = candidate_lookup.loc[true_id]

    print("=" * 70)
    print("MISSED PAIR")
    print("=" * 70)

    print("S1:", s1_id)
    print("Name:", s1_row["business_name"])
    print("Address:", s1_row["business_address"])
    print("Country:", s1_row["country"])

    print()

    print("TRUE:", true_id)
    print("Name:", true_row["business_name"])
    print("Address:", true_row["business_address"])
    print("Country:", true_row["country"])

    print()

MISSED PAIR
S1: S1-511076246
Name: Agra Granites
Address: Flat No-4, Sector-4B, Vaishno Plaza Awas Vikas Colony Sikandra, Agra, Uttar Pradesh
Country: India

TRUE: S2-100727491
Name: Agra Granites [Limited]
Address: 
Country: India

MISSED PAIR
S1: S1-210935903
Name: Leann Colon Federal Acquisitions Inc
Address: 9229 106th Way, Fl 1, Scottsdale, AZ
Country: US

TRUE: S3-121609463
Name: Leann Colon  Federal
Address: 
Country: US

MISSED PAIR
S1: S1-10580196
Name: Secure Telecom
Address: 310 Calion Street, Unit Apt C, Jonesboro, AR
Country: US

TRUE: S2-34357118
Name: SECURE TELECOM INC.
Address: 
Country: US

MISSED PAIR
S1: S1-748867572
Name: Britt Best Willow LLC
Address: 16955 Toronto Avenue, Unit Apartment 115, Prior Lake, MN
Country: US

TRUE: S3-726962279
Name: britt best willow willow llc
Address: 
Country: US

MISSED PAIR
S1: S1-651483834
Name: IX Innovative Abony Corp
Address: 40 Savage Lane, NC, Hendersonville
Country: US

TRUE: S2-453351217
Name: Corp IX Innovative Abony
Addr

In [69]:
# ============================================================
# EXPERIMENT 2
# NAME-DOMINANT MULTI-PASS BLOCKING
#
# SELF-CONTAINED SETUP
# ============================================================

import pandas as pd
import numpy as np

import re
import unicodedata

from pathlib import Path
from collections import defaultdict, Counter

from anyascii import anyascii

from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path("sampled_data")

S1_FILE = BASE_DIR / "sample_source1.tsv"
S2_FILE = BASE_DIR / "sample_source2.tsv"
S3_FILE = BASE_DIR / "sample_source3.tsv"
GT_FILE = BASE_DIR / "sample_ground_truth.tsv"

OUTPUT_DIR = Path("blocking_results")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Blocking parameters
# ------------------------------------------------------------

NAME_CHAR_TOP_K = 50
NAME_FUZZY_TOP_K = 50
NAME_TOKEN_TOP_K = 50

ADDRESS_TOP_K = 30
ADDRESS_OVERLAP_THRESHOLD = 2

NAME_CHAR_NGRAM_RANGE = (2, 5)
NAME_CHAR_MIN_DF = 1

NAME_FUZZY_THRESHOLD = 0.55


# ------------------------------------------------------------
# Weak address tokens
# ------------------------------------------------------------

WEAK_ADDRESS_TOKENS = {
    "road", "rd",
    "street", "st",
    "avenue", "ave",
    "lane", "ln",
    "drive", "dr",
    "highway", "hwy",
    "boulevard", "blvd",
    "way",
    "place", "pl",
    "parkway", "pkwy",
    "unit",
    "suite", "ste",
    "apt", "apartment",
    "floor", "fl",
    "building", "bldg",
    "block",
    "district",
    "city",
    "state",
    "county",
    "india",
    "usa",
    "us",
}


MIN_TOKEN_LENGTH = 3


# ------------------------------------------------------------
# Name legal suffixes
# ------------------------------------------------------------

LEGAL_SUFFIXES = {
    "limited",
    "ltd",
    "llc",
    "inc",
    "incorporated",
    "corp",
    "corporation",
    "company",
    "co",
    "private",
    "pvt",
    "plc",
    "llp",
    "lp",
}


print("=" * 80)
print("EXPERIMENT 2 — NAME-DOMINANT MULTI-PASS BLOCKING")
print("=" * 80)

print(f"S1 file: {S1_FILE}")
print(f"S2 file: {S2_FILE}")
print(f"S3 file: {S3_FILE}")
print(f"GT file: {GT_FILE}")

print()
print(f"Name char TF-IDF TOP-K: {NAME_CHAR_TOP_K}")
print(f"Name fuzzy TOP-K:       {NAME_FUZZY_TOP_K}")
print(f"Name token TOP-K:       {NAME_TOKEN_TOP_K}")
print(f"Address TOP-K:          {ADDRESS_TOP_K}")
print(f"Address overlap:        {ADDRESS_OVERLAP_THRESHOLD}")
print(f"Fuzzy threshold:        {NAME_FUZZY_THRESHOLD}")

EXPERIMENT 2 — NAME-DOMINANT MULTI-PASS BLOCKING
S1 file: sampled_data\sample_source1.tsv
S2 file: sampled_data\sample_source2.tsv
S3 file: sampled_data\sample_source3.tsv
GT file: sampled_data\sample_ground_truth.tsv

Name char TF-IDF TOP-K: 50
Name fuzzy TOP-K:       50
Name token TOP-K:       50
Address TOP-K:          30
Address overlap:        2
Fuzzy threshold:        0.55


In [70]:
# ============================================================
# LOAD DATA
# ============================================================

print("=" * 80)
print("LOADING DATA")
print("=" * 80)


s1 = pd.read_csv(
    S1_FILE,
    sep="\t",
    dtype=str
).fillna("")


s2 = pd.read_csv(
    S2_FILE,
    sep="\t",
    dtype=str
).fillna("")


s3 = pd.read_csv(
    S3_FILE,
    sep="\t",
    dtype=str
).fillna("")


gt = pd.read_csv(
    GT_FILE,
    sep="\t",
    dtype=str
).fillna("")


print(f"S1 shape: {s1.shape}")
print(f"S2 shape: {s2.shape}")
print(f"S3 shape: {s3.shape}")
print(f"GT shape: {gt.shape}")


print("\nColumns:")
print("S1:", list(s1.columns))
print("S2:", list(s2.columns))
print("S3:", list(s3.columns))
print("GT:", list(gt.columns))


required_source_columns = {
    "entity_id",
    "business_name",
    "business_address",
    "country"
}


for source_name, df in {
    "S1": s1,
    "S2": s2,
    "S3": s3
}.items():

    missing = (
        required_source_columns
        -
        set(df.columns)
    )

    if missing:
        raise ValueError(
            f"{source_name} is missing columns: {missing}"
        )


required_gt_columns = {
    "source1_entity_id",
    "matched_entity_ids"
}


missing_gt = (
    required_gt_columns
    -
    set(gt.columns)
)


if missing_gt:
    raise ValueError(
        f"Ground truth is missing columns: {missing_gt}"
    )


print("\nData validation passed.")

LOADING DATA
S1 shape: (1000, 4)
S2 shape: (9864, 4)
S3 shape: (10842, 4)
GT shape: (1000, 2)

Columns:
S1: ['entity_id', 'business_name', 'business_address', 'country']
S2: ['entity_id', 'business_name', 'business_address', 'country']
S3: ['entity_id', 'business_name', 'business_address', 'country']
GT: ['source1_entity_id', 'matched_entity_ids']

Data validation passed.
